In [ ]:
# Colab setup - run this first, then switch the runtime to R
# (Runtime > Change runtime type > R). Precompiled Linux binaries via Posit P3M
# keep the install to a couple of minutes instead of compiling from source.
local({
  cn <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", cn)),
          HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
})
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(
  c("SingleCellExperiment", "SummarizedExperiment", "MultiAssayExperiment",
    "S4Vectors", "basilisk"),
  update = FALSE, ask = FALSE)
install.packages(c("remotes", "reticulate", "Matrix", "ggplot2"))
remotes::install_github("PYangLab/M3", subdir = "m3-r")
# The first m3_train() builds the bundled Python engine via basilisk (a few
# minutes on first run); afterwards it is cached for the session.

# Feature attribution

Once m3 can predict disease, attribution asks which genes, cells and cell types drove that prediction. We train on the full reference, then trace each prediction back to the genes, cells, and cell types behind it.

## 1. Load the demo dataset

In [ ]:
library(m3)
library(ggplot2)
set.seed(0)
data <- m3_demo()
data

## 2. Build and train the model

Same setup as the patient-prediction tutorial, minus the held-out batch (we attribute on the full reference). `m3_train(...)` fits both the integration model and the disease predictor; attribution explains that predictor.

In [ ]:
#To save time, users can set max_epochs to 100 for test.
model <- m3_train(
  data,
  condition_keys   = c("cond_group", "Age_interval"),
  target_condition = "cond_group",
  celltype_key     = "mergedcelltype",
  batch_key        = "batch",
  donor_key        = "sample_id",
  embedding_dim    = 30L,
  max_epochs       = 500L,
  seed = 0L
)
m3_capabilities(model)

## 3. Attribute and rank the cell types

`m3_attribute(model, reference_labels = "HC")` scores how much each gene, cell, and cell type pushed the prediction away from the reference label (HC, i.e. healthy). Here we look at the **top cell types**: `m3_top_celltypes(...)` ranks them, keeping only cell types with enough cells in each condition so a small group can't mislead the ranking.

In [ ]:
attr <- m3_attribute(model, reference_labels = "HC")   # n_steps defaults to 50
cat("\ntop cell types (>= 200 cells per condition):\n")

print(m3_top_celltypes(attr, min_cells_per_condition = 200L))

## 4. Top genes

`m3_top_genes(...)` ranks the genes by how strongly they drove the prediction. `min_cells_per_condition` keeps only cell types with enough cells in each condition (so a tiny group can't dominate the ranking), and housekeeping / ribosomal genes are dropped.

In [ ]:
top100_rna <- m3_top_genes(attr, n = 100L, min_cells_per_condition = 200L, modality = "rna")
cat(sprintf("top-100 RNA genes (computed over %d balanced cell types):\n",
            top100_rna$n_celltypes_used[1]))

print(utils::head(top100_rna, 15))

**Done.** The genes, cells, and cell types behind m3's Severe-vs-HC prediction.